In [61]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from scipy import stats
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from ols_diagnostics import full_diagnostic_report, validation_report

In [62]:
df = pd.read_csv('pullman_data.csv')
df.head()

,Date,DayName,OwnPrice (Rp),CompetitorPrice (Rp),Occupancy (%)
0,2025-01-01,Wednesday,1499179,1490434,99.4
1,2025-01-02,Thursday,1755757,1466462,73.0
2,2025-01-03,Friday,1769803,1716859,65.9
3,2025-01-04,Saturday,1812768,1660975,95.3
4,2025-01-05,Sunday,1722589,1226016,82.8


In [63]:
# 1. Load & Parsing Tanggal
df['Date'] = pd.to_datetime(df['Date'])

# 2. Extract Calendar Features
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)
df['OwnPrice_centered'] = df['OwnPrice (Rp)'] - df['OwnPrice (Rp)'].mean()
df['Price_Weekend'] = df['OwnPrice_centered'] * df['IsWeekend']

df['IsFriday']   = (df['DayOfWeek'] == 4).astype(int)

m = df["Date"].dt.month
d = df["Date"].dt.day

df["IsHighSeason"] = (
    ((m == 1) & (d <= 7))
    | ((m == 6) | (m == 7))
    | ((m == 12) & (d >= 16))
).astype(int)

# 5. Tentukan Feature Set (Hindari Multikolinearitas)
feature_cols = [
    'OwnPrice (Rp)',
    'CompetitorPrice (Rp)',
    'IsWeekend',
    # 'IsFriday',
    # 'Price_Weekend',
    'IsHighSeason'
]

X = df[feature_cols]
y = df['Occupancy (%)']

# Train-Test Split (273 data pertama = Jan - Sep 2025)
X_train = X.iloc[0:273]
X_test  = X.iloc[273:]
y_train = y.iloc[0:273]
y_test  = y.iloc[273:]

In [64]:
# X_train sudah dalam bentuk DataFrame numerik (tanpa konstanta, itu ditambah otomatis)
model, results = full_diagnostic_report(
    X_train, y_train,
    feature_names=feature_cols,
    y_name="Occupancy",
    model_label="Model 2 (OwnPrice, CompetitorPrice, IsWeekend, IsHighSeason)"
)

# Validasi ke data Okt-Des
val = validation_report(model, X_test, y_test, label="Okt-Des 2025")


######################################################################
#  FULL DIAGNOSTIC REPORT - Model 2 (OwnPrice, CompetitorPrice, IsWeekend, IsHighSeason)
######################################################################

1. PEARSON CORRELATION MATRIX (antar variabel independen)
                      OwnPrice (Rp)  CompetitorPrice (Rp)  IsWeekend  IsHighSeason
OwnPrice (Rp)                 1.000                 0.429      0.329         0.321
CompetitorPrice (Rp)          0.429                 1.000      0.288         0.367
IsWeekend                     0.329                 0.288      1.000        -0.008
IsHighSeason                  0.321                 0.367     -0.008         1.000

--- Korelasi tiap X terhadap Y (Occupancy) ---
            Variabel      r  p-value Sig
       OwnPrice (Rp) 0.2582      0.0 ***
CompetitorPrice (Rp) 0.4746      0.0 ***
           IsWeekend 0.5741      0.0 ***
        IsHighSeason 0.3818      0.0 ***

2. MODEL SUMMARY (Coefficients, t-test, 

c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)


In [65]:
# 1. Cek observasi 239 (Cook's D = 0.246, jauh lebih tinggi dari model2 lain)
print(df.iloc[239])

# 2. HAC untuk Model 2
X_train_const_m2 = sm.add_constant(X_train)  # OwnPrice, CompPrice, IsWeekend, IsHighSeason
model_m2_hac = sm.OLS(y_train, X_train_const_m2).fit(cov_type='HAC', cov_kwds={'maxlags': 7})
print(model_m2_hac.summary())

Date                    2025-08-28 00:00:00
DayName                            Thursday
OwnPrice (Rp)                       2641321
CompetitorPrice (Rp)                2395691
Occupancy (%)                         100.0
Month                                     8
DayOfWeek                                 3
IsWeekend                                 0
OwnPrice_centered            1379996.838356
Price_Weekend                           0.0
IsFriday                                  0
IsHighSeason                              0
Name: 239, dtype: object
                            OLS Regression Results                            
Dep. Variable:          Occupancy (%)   R-squared:                       0.529
Model:                            OLS   Adj. R-squared:                  0.522
Method:                 Least Squares   F-statistic:                     59.72
Date:                Mon, 07 Sep 2026   Prob (F-statistic):           5.23e-36
Time:                        01:15:09   Log-Likeliho

In [66]:
import contextlib

with open("model2_report.txt", "w") as f:
    with contextlib.redirect_stdout(f):
        model, results = full_diagnostic_report(
            X_train, y_train,
            feature_names=feature_cols,
            y_name="Occupancy",
            model_label="Model 2 (OwnPrice, CompetitorPrice, IsWeekend, IsHighSeason)"
        )
        val = validation_report(model, X_test, y_test, label="Okt-Des 2025")

print("Report tersimpan")

c:\Users\ASUS\Downloads\piton\statistik\final_model\ols_diagnostics.py:186: FutureWarning: acorr_breusch_godfrey currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  bg_stat, bg_p, bg_f, bg_fp = acorr_breusch_godfrey(model, nlags=nlags)


Report tersimpan
